# Extension 2 — Intent Parser Evaluation

Evaluates the three-tier intent parser (rule-based → BERT few-shot → LLM fallback) on the 60-query test set.

**Test set breakdown:**
- 17 standard cases (clear, direct phrasings)
- 23 ambiguous cases (surface form could suggest the wrong intent)
- 20 outlier cases (indirect phrasings that rules will not match)

**Example pool:** 100 labeled queries used for BERT few-shot inference (seen during rule-based parser development, not used for evaluation).

Results are reported overall and per tier.

In [ ]:
# 1. Clone the repository
!git clone https://github.com/Voyagers-time-series-forecasting/explainable-chronos.git
%cd explainable-chronos
!git checkout fix-nli

In [ ]:
# 2. Install dependencies
!pip install --upgrade pip setuptools wheel

# If on Colab GPU, uncomment the line below:
# !pip install torch --index-url https://download.pytorch.org/whl/cu121

!pip install -e .

In [ ]:
import logging
from pathlib import Path

logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)

from extension_2.evaluation import (
    main as run_evaluation,
    print_evaluation_report,
    run_multi_turn_evaluation,
    print_multi_turn_report,
)
from extension_2.datasets import TEST_SET, EXAMPLE_SET

print(f"Example pool  : {len(EXAMPLE_SET)} queries (BERT few-shot)")
print(f"Test set      : {len(TEST_SET)} queries (reported score)")
print("Imports OK")

In [ ]:
# Run evaluation (intent-only — fast, no Chronos-2 needed)
output_directory = "/content/evaluation_results_ext2"

run_evaluation(
    seed=42,
    evaluation_set="test",
    output_dir=output_directory,
)

print(f"\nResults saved to: {output_directory}")

In [ ]:
import pandas as pd
from pathlib import Path

out = Path(output_directory)
df = pd.read_csv(out / "evaluation_results_ext2.csv")

# ── Overall metrics ────────────────────────────────────────────────────
total = len(df)
intent_acc = df["intent_correct"].mean() * 100
cov_mask = df["expected_covariate"].notna()
cov_acc = df.loc[cov_mask, "covariate_correct"].mean() * 100 if cov_mask.any() else float("nan")
tcr = df["task_completed"].mean() * 100

print(f"Test set size         : {total}")
print(f"Intent accuracy       : {intent_acc:.1f}%")
if not pd.isna(cov_acc):
    print(f"Covariate resolution  : {cov_acc:.1f}%  ({cov_mask.sum()} queries with covariate slot)")
print(f"Task completion rate  : {tcr:.1f}%")

# ── Per-tier accuracy ──────────────────────────────────────────────────
print("\n--- Accuracy by tier ---")
for tier in ("rule", "bert", "llm", "fallback"):
    mask = df["confidence_tier"] == tier
    if mask.any():
        tier_acc = df.loc[mask, "intent_correct"].mean() * 100
        print(f"  {tier:10s}: {mask.sum():3d} queries, {tier_acc:.1f}% accuracy")

# ── Per-intent breakdown ───────────────────────────────────────────────
print("\n--- Accuracy by intent type ---")
breakdown = df.groupby("expected_intent")["intent_correct"].agg(["sum", "count"])
breakdown["acc %"] = breakdown["sum"] / breakdown["count"] * 100
print(breakdown.rename(columns={"sum": "correct", "count": "total"}).to_string(float_format="%.1f"))

# ── Failed cases ───────────────────────────────────────────────────────
failed = df[~df["intent_correct"]]
if len(failed):
    print(f"\n--- Failed cases ({len(failed)}) ---")
    print(failed[["expected_intent", "parsed_intent", "confidence_tier", "query"]].to_string(index=False))
else:
    print("\nAll queries parsed correctly.")

In [ ]:
# Multi-turn state persistence
df_mt = run_multi_turn_evaluation(seed=42)
df_mt.to_csv(out / "multi_turn_results_ext2.csv", index=False)
print_multi_turn_report(df_mt)